# Data Visualization in Python with Matplotlib, Seaborn, and Streamlit

## 🧩 General Objective
- Learn how to create effective data visualizations using Matplotlib and Seaborn.
- Build a simple web application with Streamlit.
- Upload the project to GitHub to share or deploy it.


| Objective                         | Data Type                              | Recommended Chart Type                      |
|----------------------------------|----------------------------------------|---------------------------------------------|
| Distribution of a variable       | Continuous/Discrete Numerical          | Histogram, KDE, Boxplot                     |
| Compare categories               | Categorical vs. Numerical              | Boxplot, Violinplot, Barplot                |
| Compare values by category       | Nominal Categorical vs. Numerical      | Barplot, Countplot                          |
| Time trends                      | Date + Numerical Variable              | Line plot                                   |
| Correlation between two variables| Two Numerical Variables                | Scatter plot, Heatmap (for many variables)  |
| Cross between two categoricals   | Two Categorical Variables              | Heatmap, Countplot with hue                 |


In [ ]:
#!pip install streamlit

In [ ]:
import streamlit as st

st.title("Mi Primera Aplicación con Streamlit")
st.write("¡Hola mundo!")

# Agregar un widget interactivo
nombre = st.text_input("Escribe tu nombre")
if nombre:
    st.write(f"Hola, {nombre}!")

In [ ]:
# Título y texto
st.title("Título de la aplicación")
st.header("Este es un encabezado")
st.subheader("Este es un subencabezado")
st.text("Texto plano")
st.markdown("**Markdown** es _soportado_")

# Widgets interactivos
checkbox_value = st.checkbox("Mostrar gráfico")
option = st.selectbox("Elige una opción", ["Opción 1", "Opción 2"])
slider_value = st.slider("Selecciona un valor", 0, 100)

# Mostrar datos
import pandas as pd
import numpy as np

df = pd.DataFrame(np.random.randn(10, 3), columns=["A", "B", "C"])
st.dataframe(df)  # Tabla interactiva
st.table(df)      # Tabla estática

# Gráficos
import matplotlib.pyplot as plt

if checkbox_value:
    fig, ax = plt.subplots()
    ax.plot(df)
    st.pyplot(fig)

In [ ]:
# Columnas
col1, col2 = st.columns(2)
with col1:
    st.header("Columna 1")
    st.write("Contenido de la primera columna")
with col2:
    st.header("Columna 2")
    st.write("Contenido de la segunda columna")

# Pestañas
tab1, tab2 = st.tabs(["Pestaña 1", "Pestaña 2"])
with tab1:
    st.write("Contenido de la pestaña 1")
with tab2:
    st.write("Contenido de la pestaña 2")

# Expander
with st.expander("Haz clic para expandir"):
    st.write("Contenido oculto por defecto")

In [ ]:
!jupyter nbconvert --to script LAB_4_INGLES.ipynb

## 1. Quantities: 
This type of graph shows the levels of variables. Also, these graphs show the variables according to categories or classifications.

Datasets transformed into Latin characters. The following commands should be used in STATA:

cd "....\documents"

unicode analyze enaho.dta

unicode encoding set "latin1" 

unicode translate enaho.dta

In [ ]:
enaho = pd.read_stata(r"../_data/enaho.dta")
enaho

#### We will explore the BCRP data using the codes of its variables.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


In [ ]:
# Configurar Selenium (ajusta la ruta del driver)
driver = webdriver.Chrome()
driver.maximize_window()


# URL de la página que contiene los datos
url = "https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/"  
driver.get(url)

# Esperar carga

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

In [ ]:
# Esperar hasta que el input esté listo (máx 10 segundos)
wait = WebDriverWait(driver, 10)
search_box = wait.until(EC.presence_of_element_located((By.ID, "txtbuscador")))

search_box.click()
search_box.send_keys("PN01449BM")
search_box.send_keys(Keys.ENTER)  # Opcional: para simular Enter

In [ ]:
wait = WebDriverWait(driver, 10)
link_element = wait.until(EC.presence_of_element_located((By.XPATH, "//a[contains(@href, '/estadisticas/series/mensuales/resultados/') and contains(@href, 'html')]")))

# Hacer clic en el enlace
link_element.click()

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

def extraer_datos_bcrp(codigo_serie):
    # Iniciar el navegador
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    try:
        # Acceder directamente al enlace con el código proporcionado
        url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/resultados/{codigo_serie}/html"
        driver.get(url)

        # Esperar que la tabla cargue
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'table.series')))

        # Extraer filas de la tabla
        rows = driver.find_elements(By.CSS_SELECTOR, 'table.series tbody tr')
        data = []

        for row in rows:
            try:
                periodo_elem = row.find_element(By.CSS_SELECTOR, 'td.periodo b')
                dato_elem = row.find_element(By.CSS_SELECTOR, 'td.dato')

                periodo = periodo_elem.text.strip()
                dato = dato_elem.text.strip()
                data.append((periodo, dato))
            except NoSuchElementException:
                continue  # Saltar filas sin datos
        
        return data

    except TimeoutException:
        print("La página tardó demasiado en cargar o el código es inválido.")
        return []

    finally:
        driver.quit()


In [ ]:
import pandas as pd

codigo = "RD38085BM"
datos = extraer_datos_bcrp(codigo)

# Crear el DataFrame con nombres de columnas
df = pd.DataFrame(datos, columns=['Periodo', 'Dato'])

# Convertir la columna 'Dato' a numérico
df['Dato'] = pd.to_numeric(df['Dato'], errors='coerce')

print(df)

In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

def generar_fechas_bcrp(inicio, fin, como_datetime=False):
    """
    Genera una lista de fechas mensuales desde un período inicial hasta uno final.
    
    Args:
        inicio (str): Periodo inicial en formato BCRP (Ej: "Ene85")
        fin (str): Periodo final en formato BCRP (Ej: "Dic23")
        como_datetime (bool): Si True, retorna objetos datetime. Si False, retorna strings "YYYY-MM".

    Returns:
        List[str] o List[datetime]: Lista de fechas mensuales.
    """
    # Mapear los meses en español a números
    meses = {
        "Ene": 1, "Feb": 2, "Mar": 3, "Abr": 4,
        "May": 5, "Jun": 6, "Jul": 7, "Ago": 8,
        "Set": 9, "Oct": 10, "Nov": 11, "Dic": 12
    }

    def convertir_periodo(periodo):
        mes_str = periodo[:3]
        anio_str = periodo[3:]

        mes = meses[mes_str]
        anio = int(anio_str)
        if anio < 100:
            anio += 1900 if anio >= 50 else 2000  # 2 dígitos a 4

        return datetime(anio, mes, 1)

    fecha_inicio = convertir_periodo(inicio)
    fecha_fin = convertir_periodo(fin)

    fechas = []
    actual = fecha_inicio

    while actual <= fecha_fin:
        if como_datetime:
            fechas.append(actual)
        else:
            fechas.append(actual.strftime("%Y-%m"))  # o "%Y-%m-%d" si prefieres el día
        actual += relativedelta(months=1)

    return fechas

In [ ]:
fechas_dt = generar_fechas_bcrp("Ene05", "Dic22", como_datetime=True)
fechas_dt = pd.DataFrame(fechas_dt, columns=['Fecha'])
print(fechas_dt.dtypes)
fechas_dt

In [ ]:
df = df.drop(columns=['Periodo'], axis =0)
df

In [ ]:
df.set_index('fechas_dt', inplace=True)

In [ ]:
df

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time
import os

def extraer_datos_bcrp(codigos_dict):
    """
    Extrae datos del BCRP para múltiples códigos
    
    Args:
        codigos_dict: Diccionario con formato {codigo: descripcion}
    
    Returns:
        Un diccionario con los datos extraídos por cada región
    """
    # Configuración del navegador
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # Opcional: ejecutar sin interfaz gráfica
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    
    resultados = {}
    
    try:
        for codigo, descripcion in codigos_dict.items():
            print(f"Procesando {descripcion} (código: {codigo})...")
            
            # Acceder directamente al enlace con el código proporcionado
            url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/resultados/{codigo}/html"
            driver.get(url)
            
            try:
                # Esperar que la tabla cargue
                wait = WebDriverWait(driver, 10)
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'table.series')))
                
                # Extraer filas de la tabla
                rows = driver.find_elements(By.CSS_SELECTOR, 'table.series tbody tr')
                data = []
                
                for row in rows:
                    try:
                        periodo_elem = row.find_element(By.CSS_SELECTOR, 'td.periodo b')
                        dato_elem = row.find_element(By.CSS_SELECTOR, 'td.dato')
                        periodo = periodo_elem.text.strip()
                        dato = dato_elem.text.strip()
                        data.append((periodo, dato))
                    except NoSuchElementException:
                        continue  # Saltar filas sin datos
                
                resultados[descripcion] = data
                print(f"✓ Datos extraídos correctamente para {descripcion}")
                
                # Pausa breve para no sobrecargar el servidor
                time.sleep(1)
                
            except TimeoutException:
                print(f"❌ Error: Timeout al cargar datos para {descripcion} (código: {codigo})")
                resultados[descripcion] = []
    
    except Exception as e:
        print(f"Error general: {str(e)}")
    
    finally:
        driver.quit()
        
    return resultados

def guardar_resultados(resultados, formato="csv"):
    """
    Guarda los resultados en archivos
    
    Args:
        resultados: Diccionario con los datos extraídos
        formato: 'csv' o 'excel'
    """
    # Crear directorio para resultados si no existe
    os.makedirs("resultados_bcrp", exist_ok=True)
    
    # Guardar cada región en un archivo separado
    for region, datos in resultados.items():
        if not datos:
            print(f"No hay datos para guardar de {region}")
            continue
            
        df = pd.DataFrame(datos, columns=['Periodo', 'Valor'])
        
        # Limpiar el nombre del archivo
        nombre_archivo = region.replace("/", "_").replace("\\", "_").strip()
        
        if formato == "csv":
            ruta = f"resultados_bcrp/{nombre_archivo}.csv"
            df.to_csv(ruta, index=False)
        elif formato == "excel":
            ruta = f"resultados_bcrp/{nombre_archivo}.xlsx"
            df.to_excel(ruta, index=False)
            
        print(f"Archivo guardado: {ruta}")
    
    # Crear un archivo consolidado con todos los datos
    all_data = []
    
    for region, datos in resultados.items():
        if datos:
            for periodo, valor in datos:
                all_data.append((region, periodo, valor))
    
    if all_data:
        df_consolidado = pd.DataFrame(all_data, columns=['Region', 'Periodo', 'Valor'])
        
        if formato == "csv":
            df_consolidado.to_csv("resultados_bcrp/consolidado.csv", index=False)
        elif formato == "excel":
            df_consolidado.to_excel("resultados_bcrp/consolidado.xlsx", index=False)
            
        print(f"Archivo consolidado guardado en resultados_bcrp/consolidado.{formato}")

# Códigos y descripciones proporcionados
codigos_bcrp = {
    "RD38085BM": "Amazonas",
    "RD38086BM": "Ancash",
    "RD38087BM": "Apurimac",
    "RD38088BM": "Arequipa",
    "RD38089BM": "Ayacucho",
    "RD38090BM": "Cajamarca",
    "RD38091BM": "Callao",
    "RD38092BM": "Cusco",
    "RD38093BM": "Huancavelica",
    "RD38094BM": "Huánuco",
    "RD38095BM": "Ica",
    "RD38096BM": "Junín",
    "RD38097BM": "La Libertad",
    "RD38098BM": "Lambayeque",
    "RD38099BM": "Lima",
    "RD38100BM": "Loreto",
    "RD38101BM": "Madre de Dios",
    "RD38102BM": "Moquegua",
    "RD38103BM": "Pasco",
    "RD38104BM": "Piura",
    "RD38105BM": "Puno",
    "RD38106BM": "San Martín",
    "RD38107BM": "Tacna",
    "RD38108BM": "Tumbes",
    "RD38109BM": "Ucayali",
    "RD38110BM": "No registrado",
    "RD38111BM": "Total"
}

# Ejecutar el script
if __name__ == "__main__":
    print("Iniciando extracción de datos del BCRP...")
    resultados = extraer_datos_bcrp(codigos_bcrp)
    
    # Guardar resultados en CSV (puedes cambiar a "excel" si prefieres)
    guardar_resultados(resultados, formato="csv")
    
    print("Proceso completado.")

In [ ]:
resultados

In [ ]:
df = pd.DataFrame(resultados)
df

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time
import os

def extraer_datos_bcrp(codigos_dict):
    """
    Extrae datos del BCRP para múltiples códigos
    
    Args:
        codigos_dict: Diccionario con formato {codigo: descripcion}
    
    Returns:
        Un diccionario con los datos extraídos por cada región
    """
    # Configuración del navegador
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # Opcional: ejecutar sin interfaz gráfica
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    
    resultados = {}
    
    try:
        for codigo, descripcion in codigos_dict.items():
            print(f"Procesando {descripcion} (código: {codigo})...")
            
            # Acceder directamente al enlace con el código proporcionado
            url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/mensuales/resultados/{codigo}/html"
            driver.get(url)
            
            try:
                # Esperar que la tabla cargue
                wait = WebDriverWait(driver, 10)
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'table.series')))
                
                # Extraer filas de la tabla
                rows = driver.find_elements(By.CSS_SELECTOR, 'table.series tbody tr')
                periodos = []
                valores = []
                
                for row in rows:
                    try:
                        periodo_elem = row.find_element(By.CSS_SELECTOR, 'td.periodo b')
                        dato_elem = row.find_element(By.CSS_SELECTOR, 'td.dato')
                        periodo = periodo_elem.text.strip()
                        dato = dato_elem.text.strip()
                        periodos.append(periodo)
                        valores.append(dato)
                    except NoSuchElementException:
                        continue  # Saltar filas sin datos
                
                # Guardamos los datos como un diccionario con dos listas (mejor para DataFrame)
                resultados[descripcion] = {
                    'periodos': periodos,
                    'valores': valores
                }
                print(f"✓ Datos extraídos correctamente para {descripcion}")
                
                # Pausa breve para no sobrecargar el servidor
                time.sleep(1)
                
            except TimeoutException:
                print(f"❌ Error: Timeout al cargar datos para {descripcion} (código: {codigo})")
                resultados[descripcion] = {'periodos': [], 'valores': []}
    
    except Exception as e:
        print(f"Error general: {str(e)}")
    
    finally:
        driver.quit()
        
    return resultados

def guardar_resultados(resultados, formato="csv"):
    """
    Guarda los resultados en archivos
    
    Args:
        resultados: Diccionario con los datos extraídos
        formato: 'csv' o 'excel'
    """
    # Crear directorio para resultados si no existe
    os.makedirs("resultados_bcrp", exist_ok=True)
    
    # Guardar cada región en un archivo separado
    for region, datos in resultados.items():
        periodos = datos.get('periodos', [])
        valores = datos.get('valores', [])
        
        if not periodos:
            print(f"No hay datos para guardar de {region}")
            continue
            
        df = pd.DataFrame({
            'Periodo': periodos,
            'Valor': valores
        })
        
        # Limpiar el nombre del archivo
        nombre_archivo = region.replace("/", "_").replace("\\", "_").strip()
        
        if formato == "csv":
            ruta = f"resultados_bcrp/{nombre_archivo}.csv"
            df.to_csv(ruta, index=False)
        elif formato == "excel":
            ruta = f"resultados_bcrp/{nombre_archivo}.xlsx"
            df.to_excel(ruta, index=False)
            
        print(f"Archivo guardado: {ruta}")
    
    # Crear un archivo consolidado con todos los datos
    all_data = {
        'Region': [],
        'Periodo': [],
        'Valor': []
    }
    
    for region, datos in resultados.items():
        periodos = datos.get('periodos', [])
        valores = datos.get('valores', [])
        
        if periodos:
            all_data['Region'].extend([region] * len(periodos))
            all_data['Periodo'].extend(periodos)
            all_data['Valor'].extend(valores)
    
    if all_data['Region']:
        df_consolidado = pd.DataFrame(all_data)
        
        if formato == "csv":
            df_consolidado.to_csv("resultados_bcrp/consolidado.csv", index=False)
        elif formato == "excel":
            df_consolidado.to_excel("resultados_bcrp/consolidado.xlsx", index=False)
            
        print(f"Archivo consolidado guardado en resultados_bcrp/consolidado.{formato}")

# Códigos y descripciones proporcionados
codigos_bcrp = {
    "RD38085BM": "Amazonas",
    "RD38086BM": "Ancash",
    "RD38087BM": "Apurimac",
    "RD38088BM": "Arequipa",
    "RD38089BM": "Ayacucho",
    "RD38090BM": "Cajamarca",
    "RD38091BM": "Callao",
    "RD38092BM": "Cusco",
    "RD38093BM": "Huancavelica",
    "RD38094BM": "Huánuco",
    "RD38095BM": "Ica",
    "RD38096BM": "Junín",
    "RD38097BM": "La Libertad",
    "RD38098BM": "Lambayeque",
    "RD38099BM": "Lima",
    "RD38100BM": "Loreto",
    "RD38101BM": "Madre de Dios",
    "RD38102BM": "Moquegua",
    "RD38103BM": "Pasco",
    "RD38104BM": "Piura",
    "RD38105BM": "Puno",
    "RD38106BM": "San Martín",
    "RD38107BM": "Tacna",
    "RD38108BM": "Tumbes",
    "RD38109BM": "Ucayali",
    "RD38110BM": "No registrado",
    "RD38111BM": "Total"
}

# Ejecutar el script
if __name__ == "__main__":
    print("Iniciando extracción de datos del BCRP...")
    resultados = extraer_datos_bcrp(codigos_bcrp)
    
    # Guardar resultados en CSV (puedes cambiar a "excel" si prefieres)
    guardar_resultados(resultados, formato="csv")
    
    print("Proceso completado.")

In [ ]:
df = pd.DataFrame(resultados)
df